# Caption the satellite tiles with Qwen2.5-VL (PLAN Phase 4.4-4.5)

Writes one short description per 1 km tile to `/kaggle/working/captions_raw_<PROMPT_ID>.jsonl`.

**Before running:** Settings (right panel) -> *Accelerator* = **GPU T4 x2**, *Internet* = **On**,
and the dataset **gisecon-tiles** added under *Input*.

**Pilot first:** set `LIMIT = 200` in cell 3, run all, read about 30 captions. Then set `LIMIT = None` for the full run
(best as *Save Version -> Save & Run All*, so it keeps running with the browser closed).
The run is **resumable**: tiles already in the output file are skipped.

## 1. Libraries
Qwen2.5-VL needs a recent `transformers`. Takes about a minute.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate
import transformers, torch
print("transformers", transformers.__version__, "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "No GPU: set Accelerator = GPU T4 x2 in the notebook settings"

## 2. Settings
Change `LIMIT` (pilot vs full run) and `PROMPT_ID` here only.

In [ ]:
import glob, json, random, time
from pathlib import Path

MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"   # if too slow or out of memory: "Qwen/Qwen2.5-VL-3B-Instruct"
PROMPTS = {
    "base_v1": ("Describe this 1 km x 1 km satellite image in 4-6 sentences. Cover buildings (density, size, roof "
                "types), roads, green areas, water, farmland, industry or mining, and signs of human activity. Only "
                "describe what is clearly visible; do not guess place names."),
    "list_v1": ("You are looking at a 1 km x 1 km satellite image. In 4-6 short factual sentences, state: how built-up "
                "it is and what the buildings look like; the roads; vegetation and farmland; any water; any industry, "
                "mining or large facilities. Say 'none visible' for anything absent. No place names, no guesses."),
    "short_v1": ("Describe this 1 km x 1 km satellite image in at most 5 short sentences, under 90 words in total. "
                 "Cover how built-up it is and the building types, the roads, vegetation and farmland, any water, and "
                 "any industry or mining. Only describe what is clearly visible; no place names; no summary sentence."),
}
PROMPT_ID = "base_v1"
LIMIT = 200          # pilot; set to None for the full run
BATCH = 8            # lower to 4 if CUDA runs out of memory
MAX_NEW, TEMPERATURE, SEED = 200, 0.2, 42

TILES = sorted(glob.glob("/kaggle/input/**/*.png", recursive=True))
print(f"{len(TILES):,} tiles found")
assert TILES, "No PNGs under /kaggle/input: add the gisecon-tiles dataset under Input"
if LIMIT:  # pilot: a seeded random sample, not the first files (sorted by cell_id = northernmost cells)
    TILES = sorted(random.Random(SEED).sample(TILES, min(LIMIT, len(TILES))))
OUT = Path(f"/kaggle/working/captions_raw_{PROMPT_ID}.jsonl")
# resume: if an earlier run's output is attached as an input (Add Input -> Your Work), continue from it
prev = sorted(glob.glob(f"/kaggle/input/**/captions_raw_{PROMPT_ID}.jsonl", recursive=True))
if prev and not OUT.exists():
    import shutil
    shutil.copy(prev[-1], OUT)
    print(f"resuming from {prev[-1]}")

## 3. Load the model
fp16, split over both T4 GPUs. Downloading the 7B model takes a few minutes the first time.

In [ ]:
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

torch.manual_seed(SEED)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto",
                                                           attn_implementation="sdpa")
proc = AutoProcessor.from_pretrained(MODEL, max_pixels=768 * 768)
proc.tokenizer.padding_side = "left"
chat = proc.apply_chat_template([{"role": "user", "content": [{"type": "image"}, {"type": "text",
                                  "text": PROMPTS[PROMPT_ID]}]}], tokenize=False, add_generation_prompt=True)

done = {json.loads(l)["cell_id"] for l in OUT.open()} if OUT.exists() else set()
todo = [p for p in TILES if int(Path(p).stem) not in done]
print(f"{len(TILES):,} tiles, {len(done):,} already captioned, {len(todo):,} to go")

## 4. Caption
Prints the first 3 captions (check they are real sentences, not empty or `!!!!`), then progress every ~200 tiles.

In [ ]:
t0 = time.time()
with OUT.open("a") as f:
    for i in range(0, len(todo), BATCH):
        paths = todo[i:i + BATCH]
        inputs = proc(text=[chat] * len(paths), images=[Image.open(p).convert("RGB") for p in paths],
                      padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMPERATURE)
        texts = proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for p, txt in zip(paths, texts):
            f.write(json.dumps({"cell_id": int(Path(p).stem), "caption": txt.strip(), "model": MODEL,
                                "prompt_id": PROMPT_ID, "seed": SEED}) + "\n")
        f.flush()
        if i == 0:
            for p, txt in list(zip(paths, texts))[:3]:
                print(f"--- {Path(p).stem}: {txt.strip()[:300]}")
        n = i + len(paths)
        if n % 200 < BATCH or n == len(todo):
            rate = (time.time() - t0) / n
            print(f"{n}/{len(todo)}  {rate:.2f} s/image  ~{rate * (len(todo) - n) / 3600:.1f} h left", flush=True)

## 5. Check the output
Then download the `.jsonl` file from the *Output* panel (or the version's *Output* tab) and send it to the project.

In [ ]:
rows = [json.loads(l) for l in OUT.open()]
print(f"{len(rows):,} captions in {OUT.name}")
empty = sum(1 for r in rows if len(r["caption"]) < 20)
print(f"{empty} captions shorter than 20 characters")
for r in random.Random(1).sample(rows, min(5, len(rows))):
    print(f"\n[{r['cell_id']}] {r['caption']}")